# Week 1 · Day 1 — Il mio progetto: generatore di oggetti email

**Corso:** AI Engineer Core Track (LLM Engineering)

Esercizio proposto alla fine del Day 1: costruire un piccolo strumento "commerciale" basato sul caso d'uso **summarization**.

**Cosa fa:** gli passo il testo di un'email e il modello (GPT) mi propone 3 oggetti brevi, in stili diversi, più un riassunto di una riga.

**Cosa ripasso in questo notebook:**
1. caricare la chiave API da `.env`
2. scrivere un *system prompt* e uno *user prompt*
3. costruire la lista `messages`
4. chiamare `openai.chat.completions.create(...)`
5. mostrare il risultato in Markdown

## 1. Import

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

## 2. Chiave API

La chiave sta nel file `.env` (mai su GitHub! Il `.env` deve essere nel `.gitignore`).

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Nessuna chiave trovata: controlla il file .env")
elif not api_key.startswith("sk-proj-"):
    print("Chiave trovata, ma non inizia con sk-proj-: verifica che sia quella giusta")
else:
    print("Chiave API trovata!")

openai = OpenAI()
MODEL = "gpt-4.1-mini"

## 3. I prompt

- **System prompt**: chi è il modello e come deve rispondere (ruolo, tono, formato).
- **User prompt**: il contenuto concreto su cui lavorare (qui: il testo dell'email).

In [ ]:
system_prompt = """
Sei un assistente esperto di comunicazione professionale.
Ricevi il testo di un'email e proponi oggetti (subject line) efficaci.
Regole:
- Proponi esattamente 3 oggetti, ognuno di massimo 8 parole.
- Stili: 1) formale, 2) diretto, 3) amichevole.
- Aggiungi un riassunto dell'email in una sola frase.
- Rispondi nella stessa lingua dell'email.
- Rispondi in Markdown, senza blocchi di codice.
"""

user_prompt_prefix = """
Ecco il testo di un'email. Suggerisci gli oggetti e il riassunto.

"""

## 4. La funzione che costruisce i messaggi

In [ ]:
def messages_for(email_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + email_text},
    ]

## 5. La chiamata al modello

In [ ]:
def suggest_subject(email_text):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(email_text),
    )
    return response.choices[0].message.content


def display_subject(email_text):
    display(Markdown(suggest_subject(email_text)))

## 6. Proviamolo

In [ ]:
email_1 = """
Ciao Marco,
ti scrivo per ricordarti che la riunione con il cliente di giovedì è stata spostata
a venerdì alle 10:00, sempre in sala B. Porta per favore la presentazione aggiornata
con i dati del terzo trimestre e la stima dei costi del nuovo progetto.
Se hai problemi con l'orario fammi sapere entro domani.
Grazie,
Giulia
"""

display_subject(email_1)

In [ ]:
email_2 = """
Dear team,
we are happy to announce that our new customer portal will go live on Monday.
Please review the attached FAQ and let support know if you spot any issue.
Training sessions will run all next week.
Best,
Anna
"""

display_subject(email_2)

## 7. Prova con le tue email

Incolla qui il testo di una tua email.

In [ ]:
my_email = """
Scrivi qui la tua email...
"""

display_subject(my_email)

## Cosa ho imparato

- Un LLM si usa via API mandando una lista di messaggi con ruoli `system` e `user`.
- Cambiando solo il **system prompt** cambio completamente il comportamento (tono, lingua, formato).
- La risposta si legge in `response.choices[0].message.content`.
- Lo stesso schema (prompt → messages → chiamata → output) vale per qualsiasi caso di summarization: news, report, CV, email.

**Possibili estensioni:** far scegliere il tono all'utente come parametro, aggiungere un controllo di lunghezza, provare un modello gratuito con Ollama.